ModuleNotFoundError: No module named 'pandas'

In [ ]:
import pandas as pd

df = pd.read_csv("youtube_generalized_raw.csv")

ModuleNotFoundError: No module named 'pandas'

In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [10]:
import pandas as pd



In [8]:
df.shape



(43236, 10)

In [12]:
sample = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/interim/labeling_comparison_sample.csv")
disagreements = sample[~sample["agree"]]

# focus specifically on the implausible opposite-valence pairs
suspicious_pairs = [("joy", "disgust"), ("disgust", "joy"), ("sadness", "joy"),
                     ("joy", "sadness"), ("surprise", "joy"), ("joy", "surprise")]

for batch_label, single_label in suspicious_pairs:
    subset = disagreements[
        (disagreements["emotion_batch"] == batch_label) &
        (disagreements["emotion_single"] == single_label)
    ]
    print(f"\n=== batch said {batch_label}, single said {single_label} ({len(subset)} cases) ===")
    for _, row in subset.head(5).iterrows():
        print(f"  Text: {row['text_clean'][:150]}")
        print(f"  (batch: {batch_label}, single: {single_label})\n")


=== batch said joy, single said disgust (22 cases) ===
  Text: 1:13 lol anpad gavaar 'vaigyakniko ne' lol
  (batch: joy, single: disgust)

  Text: 2:30 koi tumhe rakhi badhega bhi nhi dekh ke hi lgta hai chor ho 😂😂😂
  (batch: joy, single: disgust)

  Text: I have got sasta version of the same story in my life too.😂 (Vellepanti ki bhi hadd hai like seriously fake id !!!, tum toh already fake ho (relatives
  (batch: joy, single: disgust)

  Text: Modi ji you have a large heart to beg for forgiveness in front of the nation for failing to implement such wonderful farm laws even though only a sect
  (batch: joy, single: disgust)

  Text: Bhai me gaya tha aaj tak studio ye sale sb fake dikhate hai 😂😂😂😂
  (batch: joy, single: disgust)


=== batch said disgust, single said joy (9 cases) ===
  Text: Mortien se to machhar bhi nahi marte mei kya khaak marungi 😂😂😭❤️
  (batch: disgust, single: joy)

  Text: MISHRA,  NARAYAN, UPADHAYAY,  TIWARI  PATHAK  pandit ji ra.di nachawe laglan
  (batch: disg

In [1]:
import json
from collections import Counter

# Load the JSON file
with open("/Users/harshaggarwal/Projects_4/hinemo_project/data/interim/hinglish_teaser_5k.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Count emotions
emotion_counts = Counter(item["emotion"] for item in data)

# Print results
print(emotion_counts)

Counter({'Neutral': 2611, 'Happy': 1064, 'Curious': 583, 'Frustrated': 186, 'Sad': 179, 'Humor': 140, 'Fear': 90, 'Surprised': 71, 'Angry': 62, 'Disgusted': 14})


In [2]:
file_path = "/Users/harshaggarwal/Projects_4/hinemo_project/data/Extra(fortesting)/hinglish_train (1).txt"

count = 0

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("meta"):
            count += 1

print("Number of entries:", count)

Number of entries: 15133


In [3]:


with open(file_path, "r", encoding="utf-8") as f:
    content = f.read().strip()

entries = [e.strip() for e in content.split("\n\n") if e.strip()]

seen = {}
duplicates = []

for i, entry in enumerate(entries):
    lines = entry.split("\n")

    # Ignore the meta line
    tokens = [line.split("\t")[0] for line in lines[1:] if "\t" in line]
    tweet = " ".join(tokens)

    if tweet in seen:
        duplicates.append((seen[tweet], i, tweet))
    else:
        seen[tweet] = i

print(f"Total entries: {len(entries)}")
print(f"Duplicate tweets: {len(duplicates)}")

# Show first 10 duplicates
for first, second, tweet in duplicates[:10]:
    print(f"\nDuplicate found:")
    print(f"Entry {first} and Entry {second}")
    print(tweet)

Total entries: 15131
Duplicate tweets: 0


In [4]:

ids = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("meta"):
            parts = line.strip().split("\t")
            ids.append(parts[1])

from collections import Counter

counts = Counter(ids)
duplicate_ids = {k: v for k, v in counts.items() if v > 1}

print(f"Total IDs: {len(ids)}")
print(f"Unique IDs: {len(counts)}")
print(f"Duplicate IDs: {len(duplicate_ids)}")

if duplicate_ids:
    print("Duplicate IDs:", duplicate_ids)

Total IDs: 15133
Unique IDs: 15132
Duplicate IDs: 1
Duplicate IDs: {'Eng': 2}


In [5]:
import csv
import random
import string

# Input and output files
input_file = "/Users/harshaggarwal/Projects_4/hinemo_project/data/Extra(fortesting)/hinglish_train (1).txt"
output_file = "/Users/harshaggarwal/Projects_4/hinemo_project/data/raw/twitter_takenfromsemevalpaper.csv"

# Function to generate unique 10-character IDs
used_ids = set()

def generate_id():
    while True:
        uid = ''.join(random.choices(string.ascii_letters + string.digits, k=10))
        if uid not in used_ids:
            used_ids.add(uid)
            return uid

# Read the file
with open(input_file, "r", encoding="utf-8") as f:
    content = f.read().strip()

entries = [e.strip() for e in content.split("\n\n") if e.strip()]

rows = []

for entry in entries:
    lines = entry.split("\n")

    # Skip malformed entries
    if not lines or not lines[0].startswith("meta"):
        continue

    tokens = []

    # Ignore the meta line
    for line in lines[1:]:
        parts = line.split("\t")
        if len(parts) >= 2:
            tokens.append(parts[0])

    text_clean = " ".join(tokens)

    rows.append({
        "source_id": generate_id(),
        "source": "twitter",
        "text_clean": text_clean
    })

# Save to CSV
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["source_id", "source", "text_clean"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} tweets to '{output_file}'.")

Saved 15131 tweets to '/Users/harshaggarwal/Projects_4/hinemo_project/data/raw/twitter_takenfromsemevalpaper.csv'.


In [10]:
import pandas as pd
import re

# Load the CSV
csv_file = "/Users/harshaggarwal/Projects_4/hinemo_project/data/raw/twitter_takenfromsemevalpaper.csv"
df = pd.read_csv(csv_file)

import re

def clean_text(text):
    text = str(text)

    # Remove @ mentions whether tokenized or not
    text = re.sub(r'@\s*[A-Za-z0-9_]+', '', text)

    # Remove hashtags whether tokenized or not
    text = re.sub(r'#\s*[A-Za-z0-9_]+', '', text)

    # Remove normal URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove tokenized URLs like:
    # https // t co / abc123
    # https // t . co / abc123
    text = re.sub(
        r'https?\s*/?\s*/\s*t\s*\.?\s*co\s*/\s*\S+',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove any remaining http/https tokens
    text = re.sub(r'\bhttps?\b', '', text, flags=re.IGNORECASE)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Add the new column
df["cleaned_text"] = df["text"].apply(clean_text)

# Save the updated CSV
df.to_csv(csv_file, index=False, encoding="utf-8")

print("Added 'cleaned_text' column successfully.")

Added 'cleaned_text' column successfully.


In [14]:
import pandas as pd
df = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/data/processed/sentimix_batch_labeled(15131).csv")
print(f"Total rows: {len(df)}")
print(df["emotion"].value_counts())

Total rows: 15131
emotion
joy         4318
anger       4043
neutral     3691
disgust     1897
sadness      662
fear         411
surprise     109
Name: count, dtype: int64
